In [10]:
import pandas as pd

# Load the data
job_positions = pd.read_csv('job_positions.csv')

# Filter only entries that have salary (not 'Not mentioned' and not '$0')
salary_data = job_positions[
    (job_positions['Compensation'] != 'Not mentioned') & 
    (job_positions['Compensation'] != '$0') &
    (job_positions['Compensation'].notna())
].copy()

# Clean salary values - remove $ symbol
def clean_salary(val):
    if pd.isna(val) or val == 'Not mentioned' or val == '$0':
        return None
    return float(val.replace('$', '').strip())

salary_data['Salary_Num'] = salary_data['Compensation'].apply(clean_salary)

# Remove any rows with None values
salary_data = salary_data.dropna(subset=['Salary_Num'])

# ============================================================
# 1. SALARY BY LOCATION
# ============================================================

print("="*80)
print("SALARY STATISTICS BY LOCATION")
print("="*80)

# Group by Location
location_stats = salary_data.groupby('Location')['Salary_Num'].agg([
    ('Count', 'count'),
    ('Min', 'min'),
    ('Max', 'max'),
    ('Average', 'mean'),
    ('Median', 'median'),
    ('Std', 'std')
]).round(1)

# Sort by Average descending
location_stats = location_stats.sort_values('Average', ascending=False)

# Display
display(location_stats)

# Export to CSV
location_stats.to_csv('salary_stats_by_location.csv')
print("\nSalary statistics by location exported to 'salary_stats_by_location.csv'")

# Datawrapper format
print("\nDatawrapper format (Location):")
print("-"*80)
headers = ['Location', 'Count', 'Min (K)', 'Max (K)', 'Average (K)', 'Median (K)', 'Std (K)']
print("| " + " | ".join(headers))
print("| " + " | ".join(['---'] * len(headers)))

for loc in location_stats.index:
    row = location_stats.loc[loc]
    print(f"| {loc} | {int(row['Count'])} | {row['Min']:.1f} | {row['Max']:.1f} | {row['Average']:.1f} | {row['Median']:.1f} | {row['Std']:.1f}")


# ============================================================
# 2. SALARY BY EXPERIENCE LEVEL
# ============================================================

print("\n" + "="*80)
print("SALARY STATISTICS BY EXPERIENCE LEVEL")
print("="*80)

# Group by Experience Level
experience_stats = salary_data.groupby('Experience Level')['Salary_Num'].agg([
    ('Count', 'count'),
    ('Min', 'min'),
    ('Max', 'max'),
    ('Average', 'mean'),
    ('Median', 'median'),
    ('Std', 'std')
]).round(1)

# Sort by Average descending
experience_stats = experience_stats.sort_values('Average', ascending=False)

# Display
display(experience_stats)

# Export to CSV
experience_stats.to_csv('salary_stats_by_experience.csv')
print("\nSalary statistics by experience level exported to 'salary_stats_by_experience.csv'")

# Datawrapper format
print("\nDatawrapper format (Experience Level):")
print("-"*80)
headers = ['Experience Level', 'Count', 'Min (K)', 'Max (K)', 'Average (K)', 'Median (K)', 'Std (K)']
print("| " + " | ".join(headers))
print("| " + " | ".join(['---'] * len(headers)))

for exp in experience_stats.index:
    row = experience_stats.loc[exp]
    print(f"| {exp} | {int(row['Count'])} | {row['Min']:.1f} | {row['Max']:.1f} | {row['Average']:.1f} | {row['Median']:.1f} | {row['Std']:.1f}")


# ============================================================
# 3. DETAILED VIEW - ALL SALARY ENTRIES WITH LOCATION AND EXPERIENCE
# ============================================================

print("\n" + "="*80)
print("DETAILED SALARY ENTRIES BY LOCATION AND EXPERIENCE")
print("="*80)

# Create a pivot table for better visualization
pivot_df = salary_data.groupby(['Location', 'Experience Level'])['Salary_Num'].agg([
    ('Count', 'count'),
    ('Average', 'mean')
]).round(1).reset_index()

display(pivot_df)

# Export detailed view
pivot_df.to_csv('salary_location_experience_detailed.csv', index=False)
print("\nDetailed salary by location and experience exported to 'salary_location_experience_detailed.csv'")


# ============================================================
# 4. PIVOT TABLE - LOCATION x EXPERIENCE (Average Salary)
# ============================================================

print("\n" + "="*80)
print("PIVOT TABLE - AVERAGE SALARY BY LOCATION AND EXPERIENCE LEVEL")
print("="*80)

# Create pivot table with average salary
pivot_avg = salary_data.pivot_table(
    values='Salary_Num', 
    index='Location', 
    columns='Experience Level', 
    aggfunc='mean'
).round(1)

# Sort by total average
pivot_avg['Total_Avg'] = pivot_avg.mean(axis=1)
pivot_avg = pivot_avg.sort_values('Total_Avg', ascending=False)
pivot_avg = pivot_avg.drop(columns=['Total_Avg'])

display(pivot_avg)

# Export to CSV
pivot_avg.to_csv('salary_location_experience_pivot.csv')
print("\nPivot table (average salary by location and experience) exported to 'salary_location_experience_pivot.csv'")

# Datawrapper format for pivot table
print("\nDatawrapper format (Pivot Table):")
print("-"*80)
headers = ['Location'] + [col for col in pivot_avg.columns]
print("| " + " | ".join(headers))
print("| " + " | ".join(['---'] * len(headers)))

for loc in pivot_avg.index:
    row_values = [str(val) if not pd.isna(val) else '' for val in pivot_avg.loc[loc].values]
    print("| " + " | ".join([loc] + row_values))


print("\n" + "="*80)
print("FILES EXPORTED:")
print("="*80)
print("  - salary_stats_by_location.csv")
print("  - salary_stats_by_experience.csv")
print("  - salary_location_experience_detailed.csv")
print("  - salary_location_experience_pivot.csv")

SALARY STATISTICS BY LOCATION


,Count,Min,Max,Average,Median,Std
Location,,,,,,
North America,70,73.0,404.0,193.3,195.0,62.4
Asia,7,18.0,275.0,191.0,236.0,104.6
Europe,14,3.0,263.0,122.1,94.5,81.0
Global,2,104.0,115.0,109.5,109.5,7.8



Salary statistics by location exported to 'salary_stats_by_location.csv'

Datawrapper format (Location):
--------------------------------------------------------------------------------
| Location | Count | Min (K) | Max (K) | Average (K) | Median (K) | Std (K)
| --- | --- | --- | --- | --- | --- | ---
| North America | 70 | 73.0 | 404.0 | 193.3 | 195.0 | 62.4
| Asia | 7 | 18.0 | 275.0 | 191.0 | 236.0 | 104.6
| Europe | 14 | 3.0 | 263.0 | 122.1 | 94.5 | 81.0
| Global | 2 | 104.0 | 115.0 | 109.5 | 109.5 | 7.8

SALARY STATISTICS BY EXPERIENCE LEVEL


,Count,Min,Max,Average,Median,Std
Experience Level,,,,,,
Lead,23,80.0,404.0,205.9,200.0,71.2
Senior,55,43.0,300.0,193.2,215.0,63.5
Mid-level,11,3.0,178.0,99.1,104.0,52.2
Entry-level,4,62.0,101.0,86.0,90.5,17.8



Salary statistics by experience level exported to 'salary_stats_by_experience.csv'

Datawrapper format (Experience Level):
--------------------------------------------------------------------------------
| Experience Level | Count | Min (K) | Max (K) | Average (K) | Median (K) | Std (K)
| --- | --- | --- | --- | --- | --- | ---
| Lead | 23 | 80.0 | 404.0 | 205.9 | 200.0 | 71.2
| Senior | 55 | 43.0 | 300.0 | 193.2 | 215.0 | 63.5
| Mid-level | 11 | 3.0 | 178.0 | 99.1 | 104.0 | 52.2
| Entry-level | 4 | 62.0 | 101.0 | 86.0 | 90.5 | 17.8

DETAILED SALARY ENTRIES BY LOCATION AND EXPERIENCE


,Location,Experience Level,Count,Average
0,Asia,Lead,1,263.0
1,Asia,Mid-level,1,18.0
2,Asia,Senior,5,211.2
3,Europe,Entry-level,2,81.5
4,Europe,Lead,1,80.0
5,Europe,Mid-level,1,3.0
6,Europe,Senior,10,146.4
7,Global,Mid-level,1,104.0
8,Global,Senior,1,115.0
9,North America,Entry-level,2,90.5



Detailed salary by location and experience exported to 'salary_location_experience_detailed.csv'

PIVOT TABLE - AVERAGE SALARY BY LOCATION AND EXPERIENCE LEVEL


Experience Level,Entry-level,Lead,Mid-level,Senior
Location,,,,
Asia,NaN,263.0,18.0,211.2
North America,90.5,209.2,120.6,204.9
Global,NaN,NaN,104.0,115.0
Europe,81.5,80.0,3.0,146.4



Pivot table (average salary by location and experience) exported to 'salary_location_experience_pivot.csv'

Datawrapper format (Pivot Table):
--------------------------------------------------------------------------------
| Location | Entry-level | Lead | Mid-level | Senior
| --- | --- | --- | --- | ---
| Asia |  | 263.0 | 18.0 | 211.2
| North America | 90.5 | 209.2 | 120.6 | 204.9
| Global |  |  | 104.0 | 115.0
| Europe | 81.5 | 80.0 | 3.0 | 146.4

FILES EXPORTED:
  - salary_stats_by_location.csv
  - salary_stats_by_experience.csv
  - salary_location_experience_detailed.csv
  - salary_location_experience_pivot.csv
